In [ ]:
from src.data_loader import IoTDataLoader
from src.preprocessor import IoTPreprocessor
from src.feature_engineering import IoTFeatureEngineer
from src.models import IoTModels
from src.eda import IoTEda
from src.evaluator import IoTEvaluator
from src.balancer import IoTBalancer

loader = IoTDataLoader(file_path='data\\RT_IOT2022.csv')

raw_df = loader.load_data()


In [ ]:
eda = IoTEda()

eda.run_integrated_analysis(raw_df)

In [ ]:
prep = IoTPreprocessor()

X_train, X_test, y_train, y_test = prep.fit_transform_split(raw_df)

print(X_train.shape)

In [ ]:
engineer = IoTFeatureEngineer()

X_train_fe = engineer.fit_transform(X_train)
X_test_fe = engineer.transform(X_test)

engineer.summary()
engineer.plot(raw_df, save_path="results\\feature_engineering\\fe.png")

In [ ]:
balancer = IoTBalancer(strategy="smote")

X_train_bal, y_train_bal = balancer.fit_resample(
    X_train_fe.values,
    y_train.values
)

balancer.summary()

In [ ]:
trainer = IoTModels(save_dir="results/models")
trainer.train_all(X_train_bal, y_train_bal)

In [ ]:
import pandas as pd

evaluator = IoTEvaluator(save_dir="results/evaluation")

summary = evaluator.evaluate_all(
    models=trainer.get_all_models(),
    X_test=X_test_fe.values,
    y_test=y_test.values
)

dt = trainer.get_model("Decision Tree")
fi = pd.Series(dt.feature_importances_,
               index=X_train_fe.columns).sort_values(ascending=False)

nn = trainer.get_model("Neural Network")

evaluator.plot(
    y_test=y_test.values,
    feature_importances=fi,
    loss_curve=nn.loss_curve_
)

print(summary.round(4))